# Vlasov-Ampere inverse

Execute the complete original grid with fresh degree-251 phases and inspect actual electric fields.

## Preserve the full alternating block sequence

The displayed Circuit factory composes the original 17-bit encoding on an
18-bit signal register and applies all 252 phase rotations with 251 alternating
block/adjoint applications. This is explicit FlagQuantum Circuit composition,
not automatic qfunc synthesis of a plasma equation. Host code reconstructs
the 128-dimensional block and fits the inverse response for its spectrum.

Inspect block normalization, projected field scale and success amplitude
alongside relative shape error. All eight field values and original phase
degree remain. A normalized conditional vector does not establish the cost
of obtaining that field experimentally. The block composition remains a
high-level API gap, separate from this accepted Circuit workflow.


Use the existing native runtime and plotting packages.

In [ ]:
import flagquantum as fq
import torch

from examples.synthesis.applications.plasma.vlasov_ampere.overview.quantum import (
    Model,
    encoding,
)


def circuit_preview(model: Model) -> fq.Circuit:
    result = fq.Circuit(18, dtype=torch.complex128)
    result.compose(encoding.input_circuit(model.source), qubits=tuple(range(1, 18)))
    result.h(0)
    block = encoding.block_encoding()
    inverse = block.adjoint()
    projector = fq.Circuit(18, dtype=torch.complex128)
    encoding.Builder(projector).flip(17, tuple((wire, 0) for wire in range(7, 15)))
    for index, phase in enumerate(model.phases):
        if index:
            unitary = inverse if index % 2 else block
            result.compose(unitary, qubits=tuple(range(1, 18)))
        result.compose(projector)
        result.rz(0, theta=2 * float(phase))
        result.compose(projector)
    return result.h(0)


from pathlib import Path

from IPython.display import Image, display

from examples.synthesis.applications.plasma.vlasov_ampere.overview import (
    main,
    model,
    plots,
    quantum,
    workflow,
)
from examples.synthesis.applications.plasma.vlasov_ampere.vlasov_ampere_qiskit.overview import (
    quantum as encoding,
)


Use public native Circuit composition and bounded execution.

In [ ]:
run_inverse = workflow.run


Retain all original plasma, source and 8 by 8 grid parameters.

In [ ]:
temperature, source_width, source_center, frequency = 1, 3, 50, 0.8
assert encoding.VELOCITY.numel() == 8


Keep signed three-bit velocity, three-bit position and the electric-sector bit.

In [ ]:
data_amplitudes = 128


Use the exact finite-difference derivative in the native block encoding.

In [ ]:
derivative = encoding.derivative


Normalize all four original boundary coefficients freshly.

In [ ]:
boundary = encoding.BC / encoding.BC_NORM


Preserve the boundary branch preparation, phase and flag semantics.

In [ ]:
assert boundary.numel() == 4


Keep the original derivative normalization and grid spacing.

In [ ]:
spatial_step = 100 / 7


Preserve the signed velocity ordering.

In [ ]:
print(encoding.VELOCITY.tolist())


Keep both original incoming-boundary predicates.

In [ ]:
velocity_bits, position_bits = 3, 3


Compose the full advective branch.

In [ ]:
advection_normalization = encoding.ADVECTION_NORM


Retain both E sectors and the E=0 advective selection.

In [ ]:
assert data_amplitudes == 2 ** (velocity_bits + position_bits + 1)


Build the Maxwellian-weighted force vector freshly.

In [ ]:
force = encoding.FORCE


Keep the original velocity current term and its DV factor.

In [ ]:
current_normalization = encoding.CURRENT_NORM


Preserve the two off-diagonal branches and unequal normalization.

In [ ]:
off_diagonal_normalization = encoding.OFF_DIAGONAL_NORM


Retain all force and current data positions.

In [ ]:
assert force.numel() == 8


Use all weighted branches, including the relative i-frequency phase.

In [ ]:
normalization = encoding.NORMALIZATION


Build the complete native block, including both original clean work bits.

In [ ]:
block = encoding.block_encoding()
assert block.n_qubits == 17


Reconstruct every actual projected operator column with native execution.

In [ ]:
encoded_matrix = workflow.reconstruct_block()
assert encoded_matrix.shape == (128, 128)


Generate and normalize the original Gaussian source at all eight grid positions.

In [ ]:
initial_model = model.build(encoded_matrix)
print(f"Fresh source normalization: {initial_model.source_norm}")


Use the original degree 251 and freshly fit every inverse phase.

In [ ]:
assert model.DEGREE == 251 and len(initial_model.phases) == 252
print(
    f"kappa={initial_model.kappa}, inverse alpha={initial_model.inverse_normalization}"
)


Use the complete eighteen-bit inverse program.

In [ ]:
program = circuit_preview(initial_model)
assert program.n_qubits == 18


Execute all 13,560 original native instructions with bounded state transfer.

In [ ]:
result = run_inverse(initial_model)


Read the genuine block-zero probability and full-state oracle comparison.

In [ ]:
main.report(result)


Read all eight electric-field amplitudes from the executed inverse.

In [ ]:
assert len(result.quantum_field) == 8


Preserve the source classical solve, including its explicit i source phase.

In [ ]:
assert len(result.classical_field) == 8
main.write(result, Path("vlasov-inverse-results.json"))


Plot the reconstructed quantum operator and both original field readouts.

In [ ]:
figure_directory = Path("vlasov-inverse-figures")
for filename in plots.save(result, figure_directory):
    display(Image(filename=str(figure_directory / filename)))
